# Multimodal Models- Give Your LLM Eyes

**Module 6 · Lesson 6.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Meet the Multimodal Model
- How a VLM Works - Eyes on a Brain
- Using a VLM - Image Q&A
- Document & Structured Understanding
- Beyond Single Images - Multi-Image, Video, RAG, and Limits
- The 2026 Landscape and Production

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The whole game: an LLM that can also see

> 👁️ **Analogy**
>
> **Picture the LLM you already know - a brain that reads and talks - and give it a pair of eyes.** The eyes are Lesson 6.4's vision encoder (the CLIP/SigLIP model you built there); a small "projector" is the optic nerve that translates what the eyes see into signals the brain understands; and the brain (the LLM from Modules 4-5) then talks about the picture. Send it an image plus a question, get an answer in words. That is a **vision-language model (VLM)**, or multimodal LLM.
> Everything in this lesson is that one shape: **image + text in, text out**. Describe a scene, read a receipt, compare two photos, summarize a video - all the same pattern, one model, prompted like any LLM but now with a picture attached.
> **Where the analogy breaks down:** a real optic nerve carries enormous fidelity. The projector squeezes a whole image into only a few hundred to a few thousand tokens, so fine print, exact counts, and precise positions get lost - which is exactly why VLMs miscount, misplace, and hallucinate confident detail (Step 5). Fluent is not the same as correct.

This is the pay-off of Lesson 6.4's forward hook: the CLIP/SigLIP encoder you built there becomes the VLM's eyes here. It also builds on Modules 4-5 (the LLM and prompting the VLM inherits) and Lesson 5.5 (structured output, which we reuse for reliable document extraction). One forward hook: agentic vision - a VLM that *acts* on what it sees - is a topic we'll come back to in Module 11.

- **Explain** how a VLM turns an image into tokens (a vision encoder plus a projector) and reads them alongside text in the LLM - the "eyes bolted onto a brain" architecture.

- **Apply** a modern multimodal API (the unified google-genai SDK, `gemini-3`) to image Q&A, structured document extraction, and multi-image reasoning.

- **Analyze** a VLM's failure modes - hallucinated detail, weak counting and spatial reasoning, high-res token cost - and design prompts and checks around them.

- **Compare** the 2026 VLM landscape - native vs bolted-on, closed (Gemini 3 / GPT-5) vs open (Qwen3-VL / GLM-4.5V) - and choose for a task, cost, and privacy need.

## Warm-up: 60 seconds of recall

Tap each card to check yourself. Each one is load-bearing for today.

## The setup: one client we reuse all lesson

Do not worry about the details yet - Step 1 explains the idea in plain words; this is just the `client` every example reuses. Examples use the current unified **google-genai** SDK (the older `google.generativeai` was deprecated in 2025 - [migration guide](https://ai.google.dev/gemini-api/docs/migrate)). One `client`, and an image is just another "part" you send alongside text. The same hosted-API shape works for OpenAI and Anthropic vision too.

**📝 `setup.py`** - *google-genai*

In [ ]:
# pip install google-genai pillow
from google import genai
from PIL import Image
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# An image is just another "part" in the request, next to your text.
img = Image.open("receipt.jpg")
resp = client.models.generate_content(
    model="gemini-3-flash",
    contents=["What is the total and the date on this receipt?", img],
)
print(resp.text)
# Output: The total is Rs. 1,240 and the date is 12 March 2026.

- `genai.Client(...)` - one reusable client (the deprecated package used a global `configure()`). No GPU or model download - this calls a hosted model.

- **An image is a "part"**: `contents` is a list mixing text and a PIL image. The SDK uploads and encodes the image; you just pass it alongside your question.

- `gemini-3-flash` is a fast, cheap multimodal model; `gemini-3-pro` is stronger for hard visual reasoning. The image + text in, text out shape is the whole API. (Model IDs are the mid-2026 generation - run `client.models.list()` to confirm what your key can call.)

- The same pattern holds across *hosted* APIs - OpenAI (`image_url` content) and Anthropic (image blocks) - so once you know one hosted API you know them all. (Self-hosted open models use a different local API, shown in Step 6.)

---

## 🎯 Concept 1: Meet the Multimodal Model

### Meet the Multimodal Model

One model, many questions: the same picture answers "describe it", "read the sign", "is it daytime".

**A friend describing a photo over the phone.** You can ask them anything about it - "what's in it?", "what does that sign say?", "how many people?" - and they answer in words. One friend, any question. A VLM is that friend, and you attach the photo to your message.

The gain: a VLM is an LLM with eyes - image + text in, text out. The question picks the task; no per-task model.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  A multimodal model is **one general model** you prompt with an image plus text; the question selects the task - describe, read, count, reason, compare. It is the LLM you already know, now with eyes. Every use in this lesson is that same shape, which is why one API call covers image Q&A, document reading, and multi-image analysis.

---

## 🎯 Concept 2: How a VLM Works - Eyes on a Brain

### How a VLM Works - Eyes on a Brain

Vision encoder turns the image into tokens; a projector translates them; the LLM reads them next to your words.

**Eyes, an optic nerve, and a brain.** The eyes (vision encoder) take in the picture; the optic nerve (projector) translates what they see into signals the brain speaks; the brain (LLM) thinks and talks. The projector is the new piece - a small translator between the vision world and the language world.

The gain: VLM = vision encoder (6.4) + projector + LLM. The projector maps image patches into "tokens" the LLM already understands.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

- **Vision encoder** - the CLIP/SigLIP ViT from Lesson 6.4. It slices the image into a grid of small squares ("patches") and turns each square into one vector - a *patch embedding* (a big image means many patches, so many vectors). This part you already built.

- **Projector** (the connector / adapter) - the encoder's vectors are the wrong shape for the LLM, like a plug that will not fit the socket. The projector is a small **MLP** (a multi-layer perceptron - just a little neural network with a couple of layers) that reshapes those vision embeddings into the *LLM's* token space, so they look like tokens the language model already understands. This is the genuinely new piece.

- **The LLM** - reads the projected image tokens *interleaved* with your text tokens and generates the answer, attending across both. That is why it can answer questions about the picture. (These image "tokens" are not words - they are vectors sitting in the same sequence slots as your word-tokens, so "token" here means "one item in the sequence", not "a piece of a word".)

- **Bolted-on vs native:** the LLaVA recipe bolts a pretrained encoder onto a pretrained LLM and trains mainly the projector (cheap, modular). Frontier models (Gemini, GPT-5) are *native multimodal* - trained jointly from the start, which *grounds* better. (Grounding = how tightly the model's words are tied to what is actually in the pixels; a better-grounded model invents less.)

#### 💡 What just happened

⚡ What just happened?
  A VLM is three parts: a **vision encoder** (6.4) that turns the image into patch embeddings, a **projector** that maps them into the LLM's token space, and the **LLM** that reads those image tokens next to your text and answers. **Bolted-on** models (LLaVA) train mainly the projector; **native** models (Gemini, GPT-5) train it all jointly. The image-to-few-tokens squeeze is why detail gets lost.

---

## 🎯 Concept 3: Using a VLM - Image Q&A

### Using a VLM - Image Q&A

Send the image plus a well-framed question. The prompt is where you control the answer.

**Asking a sharp intern to look at something.** "Take a look" gets a shrug; "check whether any shelf label doesn't match the product below it, and list the mismatches" gets a useful answer. Same eyes, better instruction.

The gain: prompt the image like you prompt an LLM. Role + what to look for + output format = specific answers.

**📝 `image_qa.py`** - *google-genai*

In [ ]:
# reuse `client` and PIL Image from the setup
shelf = Image.open("shelf.jpg")

prompt = (
    "You are a retail auditor. Look at this shelf photo and list every product "
    "that is out of stock or clearly misplaced. One item per line; if none, say 'all good'."
)
resp = client.models.generate_content(
    model="gemini-3-flash",
    contents=[prompt, shelf],
)
print(resp.text)
# Output: - Shelf 2: cereal slot empty (out of stock)
# Output: - Shelf 3: shampoo bottle in the snacks row (misplaced)

- Same call as the setup - `contents=[prompt, image]` - but the **prompt does the work**: a role ("retail auditor"), a specific task, and an output format ("one item per line").

- Everything from Module 5 transfers: few-shot examples, a system-style instruction, "think step by step", and asking it to say when it is unsure all improve VLM answers too.

- `gemini-3-flash` handles this; reach for `gemini-3-pro` only when the visual reasoning is genuinely hard (dense charts, subtle defects).

- Resolution/cost: the image is billed as tokens (a few hundred to a few thousand, resolution-dependent), so do not upload a 4K photo when a downscaled one answers the question.

#### 💡 What just happened

⚡ What just happened?
  Image Q&A is one call - `contents=[prompt, image]` - and the **prompt is the control surface**. A role, a specific task, and a required output format turn a vague description into an actionable list. All of Module 5's prompting carries over; model size and resolution are secondary levers.

---

## 🎯 Concept 4: Document & Structured Understanding

### Document & Structured Understanding

Read receipts, invoices, and charts into clean JSON - and let the model flag what it can't see.

**A form to fill in, not an essay to write.** Instead of "tell me about this receipt", hand the model a blank form - vendor, date, total, "legible? yes/no" - and it fills the boxes. Same-shaped answer every time, easy to store, and it can tick "can't read" instead of inventing a number.

The gain: structured output (5.5) makes document extraction reliable. Give it a schema + an "uncertain" field so it flags, not guesses.

**📝 `document_extract.py`** - *google-genai*

In [ ]:
from google.genai import types
from pydantic import BaseModel
from PIL import Image
# reuse `client` from the setup

class Receipt(BaseModel):
    vendor: str
    date: str            # ISO 8601, e.g. "2026-03-12"
    total: float
    legible: bool       # False if the model could not read it clearly

resp = client.models.generate_content(
    model="gemini-3-flash",
    contents=["Extract the receipt fields. Set legible=false if unsure.", Image.open("receipt.jpg")],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Receipt,          # force the reply to match the schema
    ),
)
data = Receipt.model_validate_json(resp.text)
print(data.vendor, data.total, data.legible)
# Output: Sunrise Grocers 1240.0 True

- `response_schema=Receipt` forces the model to return JSON matching your Pydantic schema - the same structured-output trick from Lesson 5.5, now over an image. Every receipt yields the same shape.

- The `legible` field is the key move: it lets the model **flag what it could not read** instead of hallucinating a number. Route `legible=false` records to a human. Self-reported confidence is imperfect, though, so for load-bearing totals also cross-check (a re-run or line-item math), not just trust the flag - see the anti-example below.

- Because the output is typed JSON, you validate and store it directly (`model_validate_json`) - no brittle regex over prose.

- For high-stakes fields (totals, account numbers), still verify: a VLM does structured *understanding*, not character-perfect OCR, so exact strings can be off.

#### 💡 What just happened

⚡ What just happened?
  Document AI is image Q&A plus **structured output**: ask for JSON matching a schema (Lesson 5.5) and every document yields the same shape. The critical addition is an **uncertainty field** (`legible`) so the model flags what it cannot read instead of hallucinating. A VLM does structured understanding, not perfect OCR, so verify high-stakes strings.

---

## 🎯 Concept 5: Beyond Single Images - Multi-Image, Video, RAG, and Limits

### Beyond Single Images - Multi-Image, Video, RAG, and Limits

Compare photos, sample a video, retrieve pages to feed the VLM - and know exactly where it breaks.

**Describing photos over the phone, again - but now several at once.** Your friend can compare two photos ("the second looks damaged") or skim a flip-through of video frames ("someone walks in around the middle"), but ask for an exact count or precise position and they start guessing. Same strengths, same blind spots, at scale.

The gain: multi-image and (frame-sampled) video reuse the same call. Trust holistic judgments; verify counts, positions, and tiny text.

**📝 `multi_image.py`** - *google-genai*

In [ ]:
# Multi-image: pass several images in one contents list.
# reuse `client` from the setup
from PIL import Image

before = Image.open("before.jpg")
after  = Image.open("after.jpg")

resp = client.models.generate_content(
    model="gemini-3-pro",      # pro: comparison reasoning is worth the stronger model
    contents=[
        "Image 1 is 'before', image 2 is 'after'. What changed? Be specific but "
        "say 'unsure' for anything you cannot clearly see.",
        before, after,
    ],
)
print(resp.text)
# Output: The 'after' image adds a scratch along the left door panel; the
# Output: wheel and background look unchanged. (Exact scratch length: unsure.)

- **Multi-image** is just more parts: pass several images in `contents` and label them in the prompt ("image 1 is before..."). The "say 'unsure'" instruction is doing real work against hallucination.

- **Video** is handled by sampling frames (the SDK or you extract frames at intervals) and reasoning over them - so long-video temporal detail and exact timing are weak; sampling rate matters.

- **Multimodal RAG**: retrieve the most relevant images/pages (using 6.4 embeddings in a vector database) and feed only those to the VLM - the vector-search depth is Module 8.

- **Agentic vision** - a VLM that acts on what it sees (clicks a UI, drives a workflow) - is Module 11. Here we stay at understanding, not acting.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  The same image + text call scales to **multi-image** (label the parts) and **frame-sampled video**, and it plugs into **multimodal RAG** (retrieve pages with 6.4 embeddings, feed the VLM) and, later, **agentic vision** (Module 11). But the limits are real and quiet: trust *holistic* judgments, and verify *exact* counts, positions, tiny text, and long-video timing - and always give it an "unsure" escape hatch.

---

## 🎯 Concept 6: The 2026 Landscape and Production

### The 2026 Landscape and Production

Native vs bolted-on, closed vs open - and how to choose for quality, cost, and privacy.

**Rent a car or buy one.** A closed API is a rental - top condition, zero maintenance, pay per trip, but you drive on someone else's terms. A self-hosted open model is buying - more setup, but it lives in your garage and your passengers (your data) never leave it. You choose by how much you drive and how private the cargo is.

The gain: closed = quality + ease; open self-host = privacy + cost control. Native multimodal grounds better than bolted-on.

**📝 `open_vlm.py`** - *transformers*

In [ ]:
# SELF-HOST an open VLM for private data (needs a GPU). Same idea, your infra.
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image

model_id = "Qwen/Qwen3-VL-8B-Instruct"      # open, self-hostable
model = AutoModelForImageTextToText.from_pretrained(model_id, torch_dtype="auto",
                                                    device_map="auto")
proc  = AutoProcessor.from_pretrained(model_id)

messages = [{"role": "user", "content": [
    {"type": "image", "image": Image.open("report.jpg")},
    {"type": "text",  "text": "Summarize this medical report in 3 bullet points."},
]}]
inputs = proc.apply_chat_template(messages, add_generation_prompt=True,
                                  tokenize=True, return_dict=True,
                                  return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=256)
# generate() returns prompt + new tokens, so slice the prompt off before decoding
gen = out[0][inputs["input_ids"].shape[-1]:]
print(proc.decode(gen, skip_special_tokens=True))
# Output: - Patient stable; blood pressure within the normal range ...

- Same "image + text in, text out" shape, but the model runs on *your* GPU via `transformers` - so the private documents never leave your infrastructure (`device_map="auto"` places it across available GPUs; this needs a GPU).

- `return_dict=True` makes `apply_chat_template` return the image tensors (not just `input_ids`), so `**inputs` actually carries the picture; and since `generate()` returns prompt + completion, we slice off the prompt tokens before decoding - otherwise you print your own question back.

- **Qwen3-VL** and **GLM-4.5V** are the 2026 open leaders and rival the closed models on many tasks; the smaller sizes (like 8B here) run on a single modern GPU.

- **Closed (Gemini 3 / GPT-5)** wins on raw quality and zero-ops ease; **open self-host** wins on data privacy and per-call cost at volume. Choose by the constraint that binds.

- Either way, the earlier lessons apply: prompt it well (Step 3), demand structured output (Step 4), and verify high-stakes fields - and for medical/legal use, keep a human in the loop (Module 15).

| Model | Type | Strength | Reach for it when |
|---|---|---|---|
| Gemini 3 Pro | Closed, native | Among the strongest on vision reasoning (MMMU-Pro, mid-2026) | You want the best quality, least ops |
| GPT-5(vision) | Closed, native | Strong general multimodal | You are already in the OpenAI stack |
| Qwen3-VL | Open, self-hostable | Rivals closed; long context | Privacy / cost / self-hosting |
| GLM-4.5V(Z.ai) | Open, native | Native multimodal tool use | Open + agentic / tool workflows |

#### 💡 What just happened

⚡ What just happened?
  In 2026 the frontier is **native multimodal** (trained jointly), with **Gemini 3 Pro** and **GPT-5** leading closed and **Qwen3-VL** / **GLM-4.5V** the strong open, self-hostable options. The choice is quality-and-ease (closed) versus privacy-and-cost-control (open self-host), not "newest wins". The same prompt-well / structure / verify discipline applies to all of them.

## Synthesis: one brain, new senses

### The complete picture, in one breath

A multimodal model is **an LLM with eyes**: a **vision encoder** (Lesson 6.4) turns an image into patch embeddings, a **projector** maps them into the LLM's token space, and the **LLM** reads those image tokens next to your text and answers. You use it with one shape - **image + text in, text out** - for image Q&A (prompt is the control surface), for **document extraction** (structured output + an "uncertain" field, from 5.5), and for **multi-image and sampled video**. Its limits are quiet and real - it miscounts, misplaces, and hallucinates confident detail - so you structure, constrain, and verify. In 2026 you pick **Gemini 3 / GPT-5** for quality or **Qwen3-VL / GLM-4.5V** for privacy and cost - but the "give the brain eyes" idea is the same everywhere.

> ℹ️ **Info**
>
> Where this goes next
> - **Lesson 6.6 (Video & 3D Generation)** builds on this - generating across time and space, the flip side of understanding it.
> - Multimodal RAG gets its full treatment in Module 8 - retrieve images/pages with 6.4 embeddings, then feed a VLM.
> - Agentic vision - a VLM that acts on what it sees (UIs, workflows) - is covered in Module 11.
> - Hallucination in high-stakes use (medical, KYC), bias, and human-in-the-loop are deferred for Module 15.

- Liu et al., *Visual Instruction Tuning* (LLaVA, 2023) - [arxiv.org/abs/2304.08485](https://arxiv.org/abs/2304.08485)

- Alayrac et al., *Flamingo: a Visual Language Model for Few-Shot Learning* (2022) - [arxiv.org/abs/2204.14198](https://arxiv.org/abs/2204.14198)

- Radford et al., *CLIP* (2021) - [arxiv.org/abs/2103.00020](https://arxiv.org/abs/2103.00020) (the vision encoder, Lesson 6.4)

- Bai et al., *Qwen-VL* (2023) - [arxiv.org/abs/2308.12966](https://arxiv.org/abs/2308.12966) (the Qwen3-VL open line)

- Dai et al., *InstructBLIP* (2023) - [arxiv.org/abs/2305.06500](https://arxiv.org/abs/2305.06500)

- Google Gemini API (google-genai) - [ai.google.dev/gemini-api/docs](https://ai.google.dev/gemini-api/docs) · Hugging Face transformers VLMs - [huggingface.co/docs/transformers](https://huggingface.co/docs/transformers)

## Recap

> ✅ **Info**
>
> What we covered
> - **Meet the VLM** - one model, image + text in, text out; the question picks the task.
> - **How it works** - vision encoder (6.4) + projector + LLM; bolted-on vs native.
> - **Image Q&A** - one call; the prompt is the control surface.
> - **Document extraction** - structured output (5.5) + an "uncertain" field; verify high-stakes strings.
> - **Beyond single images** - multi-image, sampled video, multimodal RAG; trust holistic, verify exact.
> - **2026 landscape** - native vs bolted-on; Gemini 3 / GPT-5 (closed) vs Qwen3-VL / GLM-4.5V (open).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Multimodal Models- Give Your LLM Eyes**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-6_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-6.5.html` - regenerate this notebook whenever the source HTML is updated.*
